In [1]:
from langchain_ollama import ChatOllama
from typing import TypedDict
from langgraph.graph import StateGraph,START,END




In [2]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=200)

In [3]:
class PriceState(TypedDict):
    order_value: float

    

In [4]:
def discount_calculator(state:PriceState):
    order_value = state['order_value']
    if order_value > 1000:
        order_value -= order_value * 0.1
    return {'order_value':order_value}

def shipping_fee_calculator(state:PriceState):
    order_value = state['order_value']
    if order_value > 500:
        order_value += 50
    return {'order_value':order_value}
    

In [ ]:
graph = StateGraph(PriceState)

graph.add_node('node_1',discount_calculator)
graph.add_node('node_2',shipping_fee_calculator)

graph.add_edge(START,'node_1')
graph.add_edge('node_1','node_2')
graph.add_edge('node_2',END)
workflow = graph.compile()

In [13]:
print(workflow.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | node_1 |   
  +--------+   
      *        
      *        
      *        
  +--------+   
  | node_2 |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [7]:
workflow.invoke({'order_value':2000})

{'order_value': 1850.0}

In [8]:
for i in workflow.stream({'order_value':2000}):
    print(i)

{'node_1': {'order_value': 1800.0}}
{'node_2': {'order_value': 1850.0}}
